# 04 — Representation Comparison, Model Selection, Hyperparameter Optimization

Compares three feature representations using 5-fold stratified cross-validation **on the TRAIN split only**
(770 subjects; VAL and TEST are never touched here):

- **A. PCA (connectome only)** — PCA fit inside each CV fold
- **B. Selected Connectome** — supervised top-K feature selection (ANOVA F-score) fit inside each CV fold, K swept over {50,100,250,500,1000}
- **C. Selected Connectome + Metadata** — best K from (B) plus quantitative (scaled) + categorical (one-hot) metadata

The connectome redundancy pruning (|Spearman| ≥ 0.7, unsupervised, no label used) is computed once on the
TRAIN split only — this never touches VAL/TEST and does not use the target, so it does not leak fold or
split information; only the *label-supervised* steps (SelectKBest, PCA-then-classifier) are refit inside
every CV fold.

In [1]:

import os, gc, time, json
os.chdir('/home/claude/adhd_project')
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score)
from xgboost import XGBClassifier

cat2 = pd.read_csv('data/processed/cat_clean.csv')
quan2 = pd.read_csv('data/processed/quan_clean.csv')
fcm2 = pd.read_csv('data/processed/fcm_clean.csv')
target2 = pd.read_csv('data/processed/target_clean.csv')
ids_train = pd.read_csv('data/processed/ids_train.csv')['participant_id'].tolist()
ids_val = pd.read_csv('data/processed/ids_val.csv')['participant_id'].tolist()
ids_test = pd.read_csv('data/processed/ids_test.csv')['participant_id'].tolist()

print('cat2', cat2.shape, 'quan2', quan2.shape, 'fcm2', fcm2.shape, 'target2', target2.shape)
print('train/val/test:', len(ids_train), len(ids_val), len(ids_test))
assert 'Sex_F' not in cat2.columns and 'Sex_F' not in quan2.columns and 'Sex_F' not in fcm2.columns


cat2 (1101, 3) quan2 (1101, 18) fcm2 (1101, 19901) target2 (1101, 2)
train/val/test: 770 165 166


## Build train/val/test row views (aligned by participant_id)

In [2]:

connectome_cols = [c for c in fcm2.columns if c != 'participant_id']

def subset(df, ids):
    d = df.set_index('participant_id').loc[ids].reset_index()
    return d

cat_train, cat_val, cat_test = subset(cat2, ids_train), subset(cat2, ids_val), subset(cat2, ids_test)
quan_train, quan_val, quan_test = subset(quan2, ids_train), subset(quan2, ids_val), subset(quan2, ids_test)
fcm_train, fcm_val, fcm_test = subset(fcm2, ids_train), subset(fcm2, ids_val), subset(fcm2, ids_test)
y_train = subset(target2, ids_train)['ADHD_Outcome'].values
y_val = subset(target2, ids_val)['ADHD_Outcome'].values
y_test = subset(target2, ids_test)['ADHD_Outcome'].values

print('Train ADHD rate: %.3f, Val: %.3f, Test: %.3f' % (y_train.mean(), y_val.mean(), y_test.mean()))


Train ADHD rate: 0.688, Val: 0.691, Test: 0.687


## Connectome redundancy pruning — fit on TRAIN only (unsupervised, no label used, never touches VAL/TEST)

In [3]:

X_conn_train_full = fcm_train[connectome_cols].astype('float32').to_numpy()
col_names = np.array(connectome_cols)

t0 = time.time()
ranks = pd.DataFrame(X_conn_train_full).rank(axis=0).to_numpy(dtype=np.float32).copy()
ranks = ranks - ranks.mean(axis=0, keepdims=True)
ranks = ranks / (ranks.std(axis=0, keepdims=True) + 1e-8)
n = ranks.shape[0]

to_drop = set()
block = 1000
for start in range(0, ranks.shape[1], block):
    end = min(start + block, ranks.shape[1])
    corr_block = (ranks[:, start:end].T @ ranks) / n
    for li, gi in enumerate(range(start, end)):
        row = corr_block[li]
        hits = np.where(np.abs(row[gi+1:]) >= 0.7)[0] + gi + 1
        for h in hits:
            to_drop.add(col_names[h])
    del corr_block
del ranks
gc.collect()

keep_cols = [c for c in connectome_cols if c not in to_drop]
print(f'Redundancy pruning done in {time.time()-t0:.1f}s. Dropped {len(to_drop)}, kept {len(keep_cols)} connectome edges.')

X_conn_train = fcm_train[keep_cols].astype('float32').to_numpy()
X_conn_val = fcm_val[keep_cols].astype('float32').to_numpy()
X_conn_test = fcm_test[keep_cols].astype('float32').to_numpy()
print('X_conn_train', X_conn_train.shape, 'X_conn_val', X_conn_val.shape, 'X_conn_test', X_conn_test.shape)


Redundancy pruning done in 10.1s. Dropped 6969, kept 12931 connectome edges.


X_conn_train (770, 12931) X_conn_val (165, 12931) X_conn_test (166, 12931)


## Metadata builders (fit-on-train-fold helper functions, used inside CV)

In [4]:

quan_cols = [c for c in quan_train.columns if c != 'participant_id']
cat_cols = [c for c in cat_train.columns if c != 'participant_id']
print('Quantitative metadata columns:', quan_cols)
print('Categorical metadata columns:', cat_cols)

def build_metadata(quan_fold_train, quan_fold_apply, cat_fold_train, cat_fold_apply):
    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    Q_train = imputer.fit_transform(quan_fold_train[quan_cols])
    Q_train = scaler.fit_transform(Q_train)
    Q_apply = scaler.transform(imputer.transform(quan_fold_apply[quan_cols]))

    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    C_train = ohe.fit_transform(cat_fold_train[cat_cols].astype(str))
    C_apply = ohe.transform(cat_fold_apply[cat_cols].astype(str))

    return np.hstack([Q_train, C_train]), np.hstack([Q_apply, C_apply])


Quantitative metadata columns: ['EHQ_EHQ_Total', 'ColorVision_CV_Score', 'APQ_P_APQ_P_CP', 'APQ_P_APQ_P_ID', 'APQ_P_APQ_P_INV', 'APQ_P_APQ_P_OPD', 'APQ_P_APQ_P_PM', 'APQ_P_APQ_P_PP', 'SDQ_SDQ_Conduct_Problems', 'SDQ_SDQ_Difficulties_Total', 'SDQ_SDQ_Emotional_Problems', 'SDQ_SDQ_Externalizing', 'SDQ_SDQ_Generating_Impact', 'SDQ_SDQ_Hyperactivity', 'SDQ_SDQ_Internalizing', 'SDQ_SDQ_Peer_Problems', 'SDQ_SDQ_Prosocial']
Categorical metadata columns: ['PreInt_Demos_Fam_Child_Ethnicity', 'PreInt_Demos_Fam_Child_Race']


## Cross-validated representation comparison (5-fold, TRAIN split only, Logistic Regression as the fast common classifier for a fair apples-to-apples comparison)

In [5]:

def cv_scores(X_builder, y, n_splits=5, seed=123, clf_builder=None):
    '''X_builder(train_idx, val_idx) -> (X_tr, X_va). clf_builder() -> fresh sklearn-compatible classifier.'''
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    accs, baccs, f1s, aucs = [], [], [], []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y)), y)):
        X_tr, X_va = X_builder(tr_idx, va_idx)
        clf = clf_builder()
        clf.fit(X_tr, y[tr_idx])
        proba = clf.predict_proba(X_va)[:, 1]
        pred = (proba >= 0.5).astype(int)
        accs.append(accuracy_score(y[va_idx], pred))
        baccs.append(balanced_accuracy_score(y[va_idx], pred))
        f1s.append(f1_score(y[va_idx], pred))
        aucs.append(roc_auc_score(y[va_idx], proba))
    return {
        'cv_accuracy_mean': np.mean(accs), 'cv_accuracy_std': np.std(accs),
        'cv_balanced_accuracy_mean': np.mean(baccs), 'cv_balanced_accuracy_std': np.std(baccs),
        'cv_f1_mean': np.mean(f1s), 'cv_f1_std': np.std(f1s),
        'cv_roc_auc_mean': np.mean(aucs), 'cv_roc_auc_std': np.std(aucs),
    }

def lr_builder():
    return LogisticRegression(penalty='l2', C=1.0, class_weight='balanced', max_iter=2000, solver='lbfgs')


In [6]:

# --- Representation A: PCA (connectome only) ---
def build_A(tr_idx, va_idx):
    pca = PCA(n_components=0.90, svd_solver='full', random_state=123)
    Xtr = pca.fit_transform(X_conn_train[tr_idx])
    Xva = pca.transform(X_conn_train[va_idx])
    return Xtr, Xva

t0 = time.time()
res_A = cv_scores(build_A, y_train, clf_builder=lr_builder)
print(f'Representation A (PCA connectome) done in {time.time()-t0:.1f}s:', res_A)


Representation A (PCA connectome) done in 8.8s: {'cv_accuracy_mean': np.float64(0.6064935064935064), 'cv_accuracy_std': np.float64(0.02950082257558584), 'cv_balanced_accuracy_mean': np.float64(0.5066823899371069), 'cv_balanced_accuracy_std': np.float64(0.023106053421537803), 'cv_f1_mean': np.float64(0.7292161467603424), 'cv_f1_std': np.float64(0.024968092498481472), 'cv_roc_auc_mean': np.float64(0.554559748427673), 'cv_roc_auc_std': np.float64(0.02007589814788782)}


In [7]:

# --- Representation B: Supervised connectome selection, sweep K, pick best by CV ROC-AUC ---
k_grid = [50, 100, 250, 500, 1000]
k_results = {}
for k in k_grid:
    def build_B(tr_idx, va_idx, k=k):
        selector = SelectKBest(f_classif, k=k)
        Xtr = selector.fit_transform(X_conn_train[tr_idx], y_train[tr_idx])
        Xva = selector.transform(X_conn_train[va_idx])
        return Xtr, Xva
    t0 = time.time()
    res_k = cv_scores(build_B, y_train, clf_builder=lr_builder)
    k_results[k] = res_k
    print(f'K={k}: CV ROC-AUC={res_k["cv_roc_auc_mean"]:.4f} (+/-{res_k["cv_roc_auc_std"]:.4f}), took {time.time()-t0:.1f}s')

best_k = max(k_results, key=lambda k: k_results[k]['cv_roc_auc_mean'])
res_B = k_results[best_k]
print(f'Best K for connectome-only selection: {best_k}')


K=50: CV ROC-AUC=0.5451 (+/-0.0483), took 0.6s


K=100: CV ROC-AUC=0.5369 (+/-0.0487), took 0.6s


K=250: CV ROC-AUC=0.5503 (+/-0.0233), took 0.6s


K=500: CV ROC-AUC=0.5566 (+/-0.0304), took 0.6s


K=1000: CV ROC-AUC=0.5629 (+/-0.0395), took 0.7s
Best K for connectome-only selection: 1000


In [8]:

# --- Representation C: Selected Connectome (best_k) + Metadata ---
def build_C(tr_idx, va_idx):
    selector = SelectKBest(f_classif, k=best_k)
    Xc_tr = selector.fit_transform(X_conn_train[tr_idx], y_train[tr_idx])
    Xc_va = selector.transform(X_conn_train[va_idx])

    quan_tr_fold = quan_train.iloc[tr_idx]
    quan_va_fold = quan_train.iloc[va_idx]
    cat_tr_fold = cat_train.iloc[tr_idx]
    cat_va_fold = cat_train.iloc[va_idx]
    Xm_tr, Xm_va = build_metadata(quan_tr_fold, quan_va_fold, cat_tr_fold, cat_va_fold)

    return np.hstack([Xc_tr, Xm_tr]), np.hstack([Xc_va, Xm_va])

t0 = time.time()
res_C = cv_scores(build_C, y_train, clf_builder=lr_builder)
print(f'Representation C (Selected Connectome K={best_k} + Metadata) done in {time.time()-t0:.1f}s:', res_C)


Representation C (Selected Connectome K=1000 + Metadata) done in 0.8s: {'cv_accuracy_mean': np.float64(0.7480519480519481), 'cv_accuracy_std': np.float64(0.027671786691769493), 'cv_balanced_accuracy_mean': np.float64(0.7029874213836478), 'cv_balanced_accuracy_std': np.float64(0.030254377297096402), 'cv_f1_mean': np.float64(0.8177486818966999), 'cv_f1_std': np.float64(0.022377963904524352), 'cv_roc_auc_mean': np.float64(0.7874606918238994), 'cv_roc_auc_std': np.float64(0.02320134917917549)}


## Representation C variant: Metadata ONLY (sanity check, given EDA showed SDQ dominance)

In [9]:

def build_D(tr_idx, va_idx):
    quan_tr_fold = quan_train.iloc[tr_idx]
    quan_va_fold = quan_train.iloc[va_idx]
    cat_tr_fold = cat_train.iloc[tr_idx]
    cat_va_fold = cat_train.iloc[va_idx]
    return build_metadata(quan_tr_fold, quan_va_fold, cat_tr_fold, cat_va_fold)

t0 = time.time()
res_D = cv_scores(build_D, y_train, clf_builder=lr_builder)
print(f'Representation D (Metadata only) done in {time.time()-t0:.1f}s:', res_D)


Representation D (Metadata only) done in 0.1s: {'cv_accuracy_mean': np.float64(0.7597402597402598), 'cv_accuracy_std': np.float64(0.03100606853587876), 'cv_balanced_accuracy_mean': np.float64(0.7570754716981133), 'cv_balanced_accuracy_std': np.float64(0.03585565046111031), 'cv_f1_mean': np.float64(0.8138822454392589), 'cv_f1_std': np.float64(0.024850776745948443), 'cv_roc_auc_mean': np.float64(0.8262971698113206), 'cv_roc_auc_std': np.float64(0.02812372952885584)}


## Representation Leaderboard

In [10]:

leaderboard = pd.DataFrame([
    {'representation': 'A: PCA (connectome only)', **res_A},
    {'representation': f'B: Selected Connectome (K={best_k})', **res_B},
    {'representation': f'C: Selected Connectome (K={best_k}) + Metadata', **res_C},
    {'representation': 'D: Metadata only', **res_D},
])
leaderboard = leaderboard.sort_values('cv_roc_auc_mean', ascending=False).reset_index(drop=True)
os.makedirs('reports', exist_ok=True)
leaderboard.to_csv('reports/representation_comparison.csv', index=False)
leaderboard[['representation','cv_accuracy_mean','cv_balanced_accuracy_mean','cv_f1_mean','cv_roc_auc_mean','cv_roc_auc_std']]


,representation,cv_accuracy_mean,cv_balanced_accuracy_mean,cv_f1_mean,cv_roc_auc_mean,cv_roc_auc_std
0,D: Metadata only,0.759740,0.757075,0.813882,0.826297,0.028124
1,C: Selected Connectome (K=1000) + Metadata,0.748052,0.702987,0.817749,0.787461,0.023201
2,B: Selected Connectome (K=1000),0.602597,0.527791,0.715413,0.562854,0.039530
3,A: PCA (connectome only),0.606494,0.506682,0.729216,0.554560,0.020076


In [11]:

winner = leaderboard.iloc[0]['representation']
print('WINNING REPRESENTATION:', winner)
print()
print('This becomes the ONLY representation carried forward. Per the project scope, we stop testing')
print('inferior representations here and optimize only the model on top of this winner.')
print()
print('SCIENTIFIC NOTE: this is a genuine, somewhat counter-intuitive finding -- properly cross-validated,')
print('leakage-safe evaluation shows the connectome (PCA or supervised-selected) barely beats chance')
print('(ROC-AUC ~0.55-0.56) at this sample size (~770 train subjects, 12,900+ edges), while the 17 SDQ/APQ')
print('behavioral questionnaire items alone reach ROC-AUC ~0.83. This matches the EDA in notebook 02, where')
print('SDQ subscales showed by far the strongest univariate correlation with ADHD_Outcome. Combining')
print('connectome with metadata (representation C) actually *hurts* versus metadata alone -- the noisy,')
print('weak connectome signal dilutes the strong behavioral signal rather than adding to it.')


WINNING REPRESENTATION: D: Metadata only

This becomes the ONLY representation carried forward. Per the project scope, we stop testing
inferior representations here and optimize only the model on top of this winner.

SCIENTIFIC NOTE: this is a genuine, somewhat counter-intuitive finding -- properly cross-validated,
leakage-safe evaluation shows the connectome (PCA or supervised-selected) barely beats chance
(ROC-AUC ~0.55-0.56) at this sample size (~770 train subjects, 12,900+ edges), while the 17 SDQ/APQ
behavioral questionnaire items alone reach ROC-AUC ~0.83. This matches the EDA in notebook 02, where
SDQ subscales showed by far the strongest univariate correlation with ADHD_Outcome. Combining
connectome with metadata (representation C) actually *hurts* versus metadata alone -- the noisy,
weak connectome signal dilutes the strong behavioral signal rather than adding to it.


## Model Selection (on the winning representation: Metadata only)

Per the project scope, at most 2 candidate models are compared, chosen by reasoning about the data:
~770 training subjects, only ~23 features after one-hot encoding (17 quantitative + 2 small-cardinality
categoricals), moderate class imbalance (69/31), and a representation that is already strongly linearly
predictive per the CV numbers above (Logistic Regression alone reached ROC-AUC 0.83).

- **Logistic Regression** — well suited to this low-dimensional, apparently near-linear signal (primary candidate)
- **XGBoost (shallow, regularized)** — tests whether nonlinear interactions between SDQ/APQ subscales add anything beyond the linear signal (backup candidate)

Both are evaluated with the same 5-fold CV on TRAIN only.

In [12]:

def xgb_builder():
    return XGBClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=3, reg_lambda=1.0, eval_metric='logloss', random_state=123, n_jobs=1
    )

res_lr = cv_scores(build_D, y_train, clf_builder=lr_builder)
res_xgb = cv_scores(build_D, y_train, clf_builder=xgb_builder)

model_comparison = pd.DataFrame([
    {'model': 'Logistic Regression (L2, balanced)', **res_lr},
    {'model': 'XGBoost (shallow, regularized)', **res_xgb},
]).sort_values('cv_roc_auc_mean', ascending=False).reset_index(drop=True)
model_comparison.to_csv('reports/model_comparison.csv', index=False)
model_comparison[['model','cv_accuracy_mean','cv_balanced_accuracy_mean','cv_f1_mean','cv_roc_auc_mean','cv_roc_auc_std']]


,model,cv_accuracy_mean,cv_balanced_accuracy_mean,cv_f1_mean,cv_roc_auc_mean,cv_roc_auc_std
0,"Logistic Regression (L2, balanced)",0.759740,0.757075,0.813882,0.826297,0.028124
1,"XGBoost (shallow, regularized)",0.775325,0.711399,0.843862,0.821934,0.030452


In [13]:

best_model_name = model_comparison.iloc[0]['model']
print('SELECTED MODEL:', best_model_name)


SELECTED MODEL: Logistic Regression (L2, balanced)


## Hyperparameter Optimization (winner model only, RandomizedSearchCV, stratified 5-fold on TRAIN)

In [14]:

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, uniform, randint

# Build the full metadata design matrix once for hyperparameter search (fit on TRAIN only)
X_meta_train, X_meta_val = build_metadata(quan_train, quan_val, cat_train, cat_val)
X_meta_train_full, X_meta_test = build_metadata(quan_train, quan_test, cat_train, cat_test)
print('X_meta_train:', X_meta_train.shape, 'X_meta_val:', X_meta_val.shape, 'X_meta_test:', X_meta_test.shape)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

if 'Logistic' in best_model_name:
    param_dist = {
        'C': loguniform(1e-3, 1e2),
        'penalty': ['l2'],
        'class_weight': ['balanced', None],
        'solver': ['lbfgs'],
    }
    base_est = LogisticRegression(max_iter=3000)
else:
    param_dist = {
        'n_estimators': randint(50, 400),
        'max_depth': randint(2, 6),
        'learning_rate': loguniform(1e-2, 3e-1),
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.6, 0.4),
        'min_child_weight': randint(1, 8),
        'reg_lambda': loguniform(1e-1, 1e1),
    }
    base_est = XGBClassifier(eval_metric='logloss', random_state=123, n_jobs=1)

search = RandomizedSearchCV(
    base_est, param_distributions=param_dist, n_iter=60, scoring='roc_auc',
    cv=skf, random_state=123, n_jobs=1, refit=True
)
t0 = time.time()
search.fit(X_meta_train, y_train)
print(f'RandomizedSearchCV done in {time.time()-t0:.1f}s')
print('Best CV ROC-AUC:', search.best_score_)
print('Best params:', search.best_params_)


X_meta_train: (770, 31) X_meta_val: (165, 31) X_meta_test: (166, 31)


RandomizedSearchCV done in 1.7s
Best CV ROC-AUC: 0.837696540880503
Best params: {'C': np.float64(0.025912819605161022), 'class_weight': None, 'penalty': 'l2', 'solver': 'lbfgs'}


In [15]:

os.makedirs('artifacts', exist_ok=True)
best_params_serializable = {k: (float(v) if isinstance(v, (np.floating,)) else int(v) if isinstance(v, (np.integer,)) else v)
                             for k, v in search.best_params_.items()}
with open('artifacts/best_hyperparameters.json', 'w') as f:
    json.dump({'model': best_model_name, 'best_cv_roc_auc': float(search.best_score_), 'params': best_params_serializable}, f, indent=2)
print(json.dumps(best_params_serializable, indent=2))


{
  "C": 0.025912819605161022,
  "class_weight": null,
  "penalty": "l2",
  "solver": "lbfgs"
}


## Threshold Optimization (using VAL set only — TEST remains untouched)

In [16]:

tuned_model = search.best_estimator_
val_proba = tuned_model.predict_proba(X_meta_val)[:, 1]

thresholds = np.linspace(0.1, 0.9, 81)
best_thr, best_f1 = 0.5, -1
for thr in thresholds:
    pred = (val_proba >= thr).astype(int)
    f1 = f1_score(y_val, pred)
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

print(f'Best threshold by VAL F1: {best_thr:.2f} (VAL F1={best_f1:.4f})')
val_pred_default = (val_proba >= 0.5).astype(int)
val_pred_tuned = (val_proba >= best_thr).astype(int)
print('VAL F1 @ 0.5:        %.4f' % f1_score(y_val, val_pred_default))
print('VAL F1 @ tuned thr:  %.4f' % f1_score(y_val, val_pred_tuned))
print('VAL ROC-AUC (threshold-independent): %.4f' % roc_auc_score(y_val, val_proba))


Best threshold by VAL F1: 0.50 (VAL F1=0.8689)
VAL F1 @ 0.5:        0.8689
VAL F1 @ tuned thr:  0.8689
VAL ROC-AUC (threshold-independent): 0.8349


In [17]:

with open('artifacts/best_hyperparameters.json') as f:
    hp = json.load(f)
hp['decision_threshold'] = float(best_thr)
with open('artifacts/best_hyperparameters.json', 'w') as f:
    json.dump(hp, f, indent=2)
print('Saved decision threshold to artifacts/best_hyperparameters.json')


Saved decision threshold to artifacts/best_hyperparameters.json
